# Chapter 1 — Sales & Revenue
Queries `mart_sales_revenue` from BigQuery and exports Plotly chart JSON for the webpage.

In [ ]:
from dotenv import load_dotenv
import os, json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from google.cloud import bigquery
from google.oauth2 import service_account

pio.json.config.default_engine = 'json'  # prevent binary encoding in exports

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id   = os.getenv('GCP_PROJECT_ID')
creds_path   = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials  = service_account.Credentials.from_service_account_file(creds_path)
client       = bigquery.Client(credentials=credentials, project=project_id)

os.makedirs(os.path.join(project_root, 'outputs'), exist_ok=True)
OUT = os.path.join(project_root, 'outputs')
print('Connected to BigQuery ✓')

In [2]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_sales_revenue`
    ORDER BY year_month
""").to_dataframe()
df.head()

,year_month,year,month,month_name,total_orders,total_items_sold,total_revenue,total_freight,total_gmv,avg_item_price,avg_order_value,unique_products_sold,active_sellers
0,2016-09,2016,9,September,2,5,207.86,71.83,279.69,41.57,136.23,3,2
1,2016-10,2016,10,October,290,346,44633.99,6901.35,51535.34,129.00,194.81,260,133
2,2016-12,2016,12,December,1,1,10.90,8.72,19.62,10.90,19.62,1,1
3,2017-01,2017,1,January,787,964,120873.30,17041.89,137915.19,125.39,193.18,612,226
4,2017-02,2017,2,February,1718,1947,246219.35,38760.28,284979.63,126.46,172.72,1256,425


In [3]:
# Chart 1 — Monthly Revenue Trend
fig1 = px.line(
    df, x='year_month', y='total_revenue',
    title='Monthly Revenue (BRL)',
    labels={'year_month': 'Month', 'total_revenue': 'Revenue (BRL)'},
    markers=True,
    color_discrete_sequence=['#2ecc71']
)
fig1.update_layout(template='plotly_white', hovermode='x unified')
fig1.show()
with open(os.path.join(OUT, 'sales_monthly_revenue.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported sales_monthly_revenue.json')

Exported sales_monthly_revenue.json


In [4]:
# Chart 2 — Monthly Orders Count
fig2 = px.bar(
    df, x='year_month', y='total_orders',
    title='Monthly Order Volume',
    labels={'year_month': 'Month', 'total_orders': 'Orders'},
    color_discrete_sequence=['#3498db']
)
fig2.update_layout(template='plotly_white')
fig2.show()
with open(os.path.join(OUT, 'sales_monthly_orders.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported sales_monthly_orders.json')

Exported sales_monthly_orders.json


In [5]:
# Chart 3 — Average Order Value over time
fig3 = px.line(
    df, x='year_month', y='avg_order_value',
    title='Average Order Value Over Time (BRL)',
    labels={'year_month': 'Month', 'avg_order_value': 'Avg Order Value (BRL)'},
    markers=True,
    color_discrete_sequence=['#e67e22']
)
fig3.update_layout(template='plotly_white', hovermode='x unified')
fig3.show()
with open(os.path.join(OUT, 'sales_avg_order_value.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported sales_avg_order_value.json')

Exported sales_avg_order_value.json


In [6]:
# Chart 4 — Active Sellers per Month
fig4 = px.bar(
    df, x='year_month', y='active_sellers',
    title='Active Sellers Per Month',
    labels={'year_month': 'Month', 'active_sellers': 'Active Sellers'},
    color_discrete_sequence=['#9b59b6']
)
fig4.update_layout(template='plotly_white')
fig4.show()
with open(os.path.join(OUT, 'sales_active_sellers.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported sales_active_sellers.json')

Exported sales_active_sellers.json
